# 🔄 Convertisseur .cha → JSONL (avec fusion par speaker)

**Rôle :** Transformer des fichiers CHILDES `.cha` + audio en paires `{audio_path, text}` au format JSONL fusionnées par speaker (10–30 s), prêtes à être consommées par le pipeline `04_whisper_preprocessing_augmentation_pipeline.ipynb`.

```
INPUT
  ├── data/cha/   → fichiers .cha  (transcriptions CHILDES avec timestamps %wor)
  └── data/audio/ → fichiers audio (.wav / .mp3 / .m4a ...)

PIPELINE
  Zone 0 : Installation & Configuration
  Zone 1 : Parsing .cha → WorSegments (word-level timestamps)
  Zone 2 : Matching .cha ↔ audio (chemin relatif)
  Zone 3 : Découpage audio individuel (FFmpeg, max_segments)
  Zone 4 : Conversion → AudioSegmentMeta
  Zone 5 : Merge contrôlé par speaker (10–30 s, silence-padded)
  Zone 6 : Quality Analysis (score 0–1, dashboard)
  Zone 7 : Export JSONL → input_pairs.jsonl

OUTPUT
  └── input_pairs.jsonl  → {id, audio_path, text, speaker, source_file, ...}
```


---
## Zone 0 : Dépendances & Configuration

In [ ]:
!apt-get install -y ffmpeg > /dev/null 2>&1
!pip install -q tqdm pandas numpy librosa soundfile
print('✅ Prêt')

In [ ]:
import re
import json
import subprocess
import tempfile
import shutil
from pathlib import Path
from typing import List, Dict, Tuple, Optional
from dataclasses import dataclass, field
from collections import Counter

import numpy as np
import pandas as pd
from tqdm import tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
#  CONFIGURATION — Adapter selon ton environnement
# ═══════════════════════════════════════════════════════════════════════════

CONFIG = {
    # ── Entrée ───────────────────────────────────────────────────────────────
    "cha_dir"          : Path("/content/drive/MyDrive/asr/data/cha"),
    "audio_dir"        : Path("/content/drive/MyDrive/asr/data/songs"),
    "audio_extensions" : [".wav", ".mp3", ".m4a", ".flac"],

    # ── Segmentation individuelle ─────────────────────────────────────────────
    # None = tous les segments bruts, ou un entier pour limiter
    "max_segments"     : 800,

    # ── Sortie ───────────────────────────────────────────────────────────────
    # Dossier temporaire des segments individuels (avant merge)
    "segments_dir"     : Path("/content/drive/MyDrive/asr/output/cha_segments"),
    # Dossier des segments fusionnés (audio + .cha par speaker)
    "merged_dir"       : Path("/content/drive/MyDrive/asr/output/merged_segments"),
    # Fichier JSONL final pour le pipeline 04
    "output_jsonl"     : Path("/content/drive/MyDrive/asr/output/input_pairs.jsonl"),

    # ── Audio ────────────────────────────────────────────────────────────────
    "sample_rate"      : 16000,

    # ── Filtres pré-segmentation ──────────────────────────────────────────────
    "min_duration_ms"  : 500,
    "max_duration_ms"  : 30000,
    "min_words"        : 1,
    # None = tous les speakers, ex: {"KAT", "WIL"} pour filtrer
    "target_speakers"  : None,

    # ── Merge contrôlé par speaker ────────────────────────────────────────────
    # Durée cible speech-only pour le merge (secondes)
    "merge_min_sec"    : 10.0,
    "merge_max_sec"    : 30.0,

    # ── Quality filtering ─────────────────────────────────────────────────────
    "quality_threshold": 0.70,   # Garder segments avec score >= seuil
}

for key in ["segments_dir", "merged_dir"]:
    CONFIG[key].mkdir(parents=True, exist_ok=True)
CONFIG["output_jsonl"].parent.mkdir(parents=True, exist_ok=True)

print('✅ Configuration chargée')
print(f'   .cha dir    : {CONFIG["cha_dir"]}')
print(f'   audio dir   : {CONFIG["audio_dir"]}')
print(f'   output JSONL: {CONFIG["output_jsonl"]}')
print(f'   max_segments: {CONFIG["max_segments"]}')
print(f'   merge cible : {CONFIG["merge_min_sec"]}–{CONFIG["merge_max_sec"]} s')

---
## Zone 1 : Parsing des fichiers .cha

Extrait les segments `%wor` avec timestamps word-level de chaque fichier CHILDES.

In [ ]:
@dataclass
class ChaSegment:
    """Tour de parole extrait d'un fichier .cha avec timestamps word-level."""
    speaker  : str
    text     : str
    words    : List[Tuple[str, int, int]]  # (word, start_ms, end_ms)
    file_name: str
    rel_path : str = ""   # chemin relatif depuis cha_dir (ex: '1/01-1a')

    @property
    def start_ms(self) -> int:  return self.words[0][1]
    @property
    def end_ms(self) -> int:    return self.words[-1][2]
    @property
    def duration_ms(self) -> int: return self.end_ms - self.start_ms
    @property
    def num_words(self) -> int:  return len(self.words)

In [ ]:
_PUNCT_TOKENS  = {'?', '.', ',', '!', '+...', '0', 'xxx', 'yyy', 'www'}
_CHA_ANNO_RE   = re.compile(r'[&@\[\]<>]')
_TIMESTAMP_RE  = re.compile(r'^\d{4,}_\d{4,}$')


def _clean_word(token: str) -> Optional[str]:
    if token in _PUNCT_TOKENS:
        return None
    cleaned = _CHA_ANNO_RE.sub('', token).strip()
    cleaned = re.sub(r'-+$', '', cleaned).strip()
    return cleaned if cleaned else None


def parse_cha_file(cha_path: Path, cha_dir: Path) -> List[ChaSegment]:
    """
    Parse un fichier .cha CHILDES et retourne la liste des ChaSegment.
    Gère les 3 formats de timestamps CHILDES du tier %wor.
    """
    segments  = []
    cur_spk   = None
    file_name = cha_path.stem
    rel_path  = str(cha_path.relative_to(cha_dir).with_suffix(''))

    try:
        lines = cha_path.read_text(encoding='utf-8', errors='replace').splitlines()
    except Exception as e:
        print(f'  ⚠️  Impossible de lire {cha_path.name}: {e}')
        return []

    for line in lines:
        line = line.rstrip()
        if line.startswith('*'):
            parts = line.split(':', 1)
            cur_spk = parts[0].replace('*', '').strip()
            continue
        if not line.startswith('%wor:') or not cur_spk:
            continue

        content = line.split(':', 1)[1].strip()
        content = re.sub(r'[\x00-\x1f]', ' ', content)
        tokens  = content.split()

        words    = []
        pending_word = None
        pending_start = None

        for tok in tokens:
            # Format 1 : mot_START_END
            m = re.match(r'^(.+?)_(\d{4,})_(\d{4,})$', tok)
            if m:
                w = _clean_word(m.group(1))
                if w:
                    words.append((w, int(m.group(2)), int(m.group(3))))
                pending_word = None
                continue

            # Format 2 : timestamp isolé START_END
            if _TIMESTAMP_RE.match(tok):
                parts_ts = tok.split('_')
                if len(parts_ts) == 2 and pending_word:
                    try:
                        words.append((pending_word, int(parts_ts[0]), int(parts_ts[1])))
                    except ValueError:
                        pass
                pending_word = None
                continue

            # Format 3 : mot seul (attend le timestamp suivant)
            cleaned = _clean_word(tok)
            if cleaned:
                pending_word = cleaned

        if len(words) >= 1:
            text = ' '.join(w[0] for w in words)
            segments.append(ChaSegment(
                speaker=cur_spk, text=text, words=words,
                file_name=file_name, rel_path=rel_path
            ))

    return segments


def parse_all_cha(cha_dir: Path) -> List[ChaSegment]:
    """Parse tous les .cha d'un dossier récursivement."""
    print('=' * 70)
    print('ZONE 1 : PARSING .CHA')
    print('=' * 70)

    cha_files = sorted(cha_dir.rglob('*.cha'))
    if not cha_files:
        print(f'❌ Aucun fichier .cha trouvé dans {cha_dir}')
        return []

    all_segments = []
    for i, f in enumerate(tqdm(cha_files, desc='Parsing .cha')):
        all_segments.extend(parse_cha_file(f, cha_dir))
        if (i + 1) % 50 == 0:
            print(f'   ✓ {i + 1}/{len(cha_files)} fichiers')

    print(f'\n   Fichiers .cha     : {len(cha_files)}')
    print(f'   Segments extraits : {len(all_segments)}')
    print(f'   Speakers uniques  : {len(set(s.speaker for s in all_segments))}')
    return all_segments


all_segments = parse_all_cha(CONFIG['cha_dir'])

---
## Zone 2 : Matching .cha ↔ Audio

Association par chemin relatif (respecte les sous-dossiers), fallback par stem.

In [ ]:
def match_audio_files(cha_dir: Path, audio_dir: Path,
                      extensions: List[str]) -> Dict[str, Path]:
    """
    Construit un dictionnaire rel_path → audio_path.
    Clé primaire : chemin relatif sans extension (ex: '1/01-1a').
    Fallback : stem seul (ex: '01-1a').
    """
    audio_map = {}
    for ext in extensions:
        for audio_path in audio_dir.rglob(f'*{ext}'):
            rel = str(audio_path.relative_to(audio_dir).with_suffix(''))
            audio_map[rel]             = audio_path   # clé relative
            audio_map[audio_path.stem] = audio_path   # fallback stem

    return audio_map


print('=' * 70)
print('ZONE 2 : MATCHING .CHA ↔ AUDIO')
print('=' * 70)

audio_map  = match_audio_files(CONFIG['cha_dir'], CONFIG['audio_dir'],
                                CONFIG['audio_extensions'])

cha_rel_paths = set(s.rel_path  for s in all_segments)
cha_stems     = set(s.file_name for s in all_segments)

matched   = cha_rel_paths & set(audio_map.keys())
missing   = cha_rel_paths - set(audio_map.keys())

print(f'\n   Fichiers .cha uniques   : {len(cha_rel_paths)}')
print(f'   Fichiers audio indexés  : {len(set(audio_map.values()))}')
print(f'   ✅ Matchés               : {len(matched)}')
if missing:
    print(f'   ⚠️  Sans audio ({len(missing)})   : {sorted(missing)[:10]}')

---
## Zone 3 : Découpage audio individuel

Extrait chaque segment via FFmpeg. Limite configurable `max_segments` (segments valides uniquement).

In [ ]:
def extract_segment_ffmpeg(audio_path: Path, start_ms: int, end_ms: int,
                            out_path: Path, sr: int = 16000) -> bool:
    """Extrait un segment audio via ffmpeg. Retourne True si succès."""
    if out_path.exists():
        return True
    cmd = [
        'ffmpeg', '-i', str(audio_path),
        '-ss', str(start_ms / 1000.0),
        '-t',  str((end_ms - start_ms) / 1000.0),
        '-ar', str(sr), '-ac', '1', '-acodec', 'pcm_s16le',
        '-y',  str(out_path)
    ]
    try:
        subprocess.run(cmd, check=True, capture_output=True, timeout=15)
        return True
    except Exception:
        return False


def apply_prefilters(seg: ChaSegment, cfg: Dict) -> Optional[str]:
    if seg.duration_ms < cfg['min_duration_ms']:
        return f'too_short ({seg.duration_ms}ms)'
    if seg.duration_ms > cfg['max_duration_ms']:
        return f'too_long ({seg.duration_ms}ms)'
    if seg.num_words < cfg['min_words']:
        return f'too_few_words ({seg.num_words})'
    if cfg['target_speakers'] and seg.speaker not in cfg['target_speakers']:
        return f'speaker_excluded ({seg.speaker})'
    return None


def segment_audio(segments: List[ChaSegment], audio_map: Dict[str, Path],
                  cfg: Dict) -> List[Dict]:
    """
    Découpe les segments individuels avec FFmpeg.
    Retourne une liste de dicts compatibles avec Zone 4 (AudioSegmentMeta).
    Respecte la limite max_segments sur les segments valides uniquement.
    """
    print('=' * 70)
    print('ZONE 3 : DÉCOUPAGE AUDIO INDIVIDUEL')
    print('=' * 70)

    out_dir      = cfg['segments_dir']
    sr           = cfg['sample_rate']
    max_segments = cfg.get('max_segments', None)
    results      = []
    stats        = {'ok': 0, 'prefilter': 0, 'no_audio': 0, 'ffmpeg_fail': 0}

    if max_segments is not None:
        print(f'\n   Limite : {max_segments} segments valides')
    else:
        print(f'\n   Pas de limite (tous les segments)')

    for i, seg in enumerate(tqdm(segments, desc='Découpage')):
        # Arrêt dès que la limite est atteinte
        if max_segments is not None and stats['ok'] >= max_segments:
            print(f'\n   🛑 Limite de {max_segments} atteinte — arrêt.')
            break

        # Filtres légers
        reject = apply_prefilters(seg, cfg)
        if reject:
            stats['prefilter'] += 1
            continue

        # Trouver l'audio (chemin relatif en priorité, stem en fallback)
        audio_path = audio_map.get(seg.rel_path) or audio_map.get(seg.file_name)
        if audio_path is None:
            stats['no_audio'] += 1
            continue

        # Créer sous-dossier speaker
        spk_dir = out_dir / seg.speaker
        spk_dir.mkdir(exist_ok=True)

        seg_id   = f'{seg.file_name}_{seg.speaker}_{i:05d}'
        out_path = spk_dir / f'{seg_id}.wav'

        if not extract_segment_ffmpeg(audio_path, seg.start_ms, seg.end_ms, out_path, sr):
            stats['ffmpeg_fail'] += 1
            continue

        results.append({
            'segment_id'   : seg_id,
            'speaker'      : seg.speaker,
            'file_name'    : seg.file_name,
            'rel_path'     : seg.rel_path,
            'audio_file'   : audio_path,          # fichier SOURCE (pour le merge)
            'segment_path' : out_path,            # segment découpé
            'mfa_start_ms' : float(seg.start_ms),
            'mfa_end_ms'   : float(seg.end_ms),
            'text'         : seg.text,
            'word_times'   : [{'word': w[0], 'start_ms': w[1], 'end_ms': w[2]}
                               for w in seg.words],
        })
        stats['ok'] += 1

    print(f'\n   Segments en entrée      : {len(segments)}')
    print(f'   ✅ Extraits avec succès  : {stats["ok"]}')
    print(f'   ⏭️  Filtrés               : {stats["prefilter"]}')
    print(f'   ⚠️  Audio manquant        : {stats["no_audio"]}')
    print(f'   ❌ Erreur ffmpeg          : {stats["ffmpeg_fail"]}')

    # Stats par speaker
    by_spk = Counter(r['speaker'] for r in results)
    print(f'   📊 Par speaker           : {dict(by_spk)}')
    return results


audio_segments = segment_audio(all_segments, audio_map, CONFIG)

---
## Zone 4 : Conversion → AudioSegmentMeta

Convertit les dicts de Zone 3 en objets `AudioSegmentMeta` compatibles avec l'algorithme de merge (Zone 5).

In [ ]:
@dataclass
class AudioSegmentMeta:
    """Segment individuel enrichi, prêt pour le merge contrôlé."""
    segment_id      : str
    speaker         : str
    file_name       : str
    rel_path        : str
    audio_file      : Path       # fichier audio SOURCE complet
    mfa_start_ms    : float      # début dans le fichier source
    mfa_end_ms      : float      # fin dans le fichier source
    text            : str
    word_times      : List[Dict] # [{'word', 'start_ms', 'end_ms'}]
    segment_audio_path: Path     # segment individuel déjà découpé

    @property
    def mfa_duration_ms(self) -> float:
        return self.mfa_end_ms - self.mfa_start_ms

    @property
    def duration_sec(self) -> float:
        return self.mfa_duration_ms / 1000.0


def convert_to_meta(audio_segments_dicts: List[Dict]) -> List[AudioSegmentMeta]:
    """Convertit les dicts de Zone 3 en AudioSegmentMeta."""
    print('=' * 70)
    print('ZONE 4 : CONVERSION AudioSegmentMeta')
    print('=' * 70)

    metas = []
    for d in audio_segments_dicts:
        metas.append(AudioSegmentMeta(
            segment_id       = d['segment_id'],
            speaker          = d['speaker'],
            file_name        = d['file_name'],
            rel_path         = d['rel_path'],
            audio_file       = Path(d['audio_file']),
            mfa_start_ms     = d['mfa_start_ms'],
            mfa_end_ms       = d['mfa_end_ms'],
            text             = d['text'],
            word_times       = d['word_times'],
            segment_audio_path = Path(d['segment_path']),
        ))

    durations = [m.duration_sec for m in metas]
    print(f'   ✅ Convertis : {len(metas)} segments')
    print(f'   📊 Par speaker : {dict(Counter(m.speaker for m in metas))}')
    if durations:
        print(f'   📊 Durée moy : {np.mean(durations):.2f}s  '
              f'| min {np.min(durations):.2f}s  | max {np.max(durations):.2f}s')
        short = sum(1 for d in durations if d < 10)
        ok    = sum(1 for d in durations if 10 <= d <= 30)
        long_ = sum(1 for d in durations if d > 30)
        print(f'   📊 <10s: {short}  | 10–30s: {ok}  | >30s: {long_}')
    return metas


audio_segment_metas = convert_to_meta(audio_segments)

---
## Zone 5 : Merge contrôlé par speaker (10–30 s)

Fusionne les segments consécutifs **du même speaker / même fichier source** en clips de 10–30 s de parole effective.

- **Audio :** `ffmpeg concat` avec silence-padding aux intervalles inter-locuteurs.
- **Timestamps :** recalculés relatifs au début du fichier produit.
- **Ne traverse jamais** une frontière de speaker ni de fichier source.


In [ ]:
@dataclass
class MergedSegment:
    """Résultat d'un merge contrôlé — audio silence-padded."""
    merge_id              : str
    speaker               : str
    file_name             : str
    rel_path              : str
    original_segment_ids  : List[str]
    texts                 : List[str]
    merged_text           : str
    merged_duration_ms    : float   # speech + silences inter-segments
    word_times_merged     : List[Dict]  # timestamps RELATIFS au fichier produit
    source_segments       : List   # List[AudioSegmentMeta]
    audio_source_file     : Path
    merged_audio_path     : Optional[Path] = None

    @property
    def duration_sec(self) -> float:  return self.merged_duration_ms / 1000.0
    @property
    def num_words(self) -> int:       return len(self.word_times_merged)

    def to_jsonl_dict(self) -> Dict:
        return {
            'id'          : self.merge_id,
            'audio_path'  : str(self.merged_audio_path),
            'text'        : self.merged_text,
            'speaker'     : self.speaker,
            'source_file' : self.file_name,
            'source_audio': str(self.audio_source_file),
            'duration_sec': round(self.duration_sec, 3),
            'num_words'   : self.num_words,
            'word_times'  : self.word_times_merged,
        }


# ─── Helpers FFmpeg ──────────────────────────────────────────────────────────

def _extract_to_temp(source_file: Path, start_ms: float, end_ms: float,
                     tmp_dir: str, idx: int, sr: int = 16000) -> Optional[Path]:
    out = Path(tmp_dir) / f'seg_{idx:04d}.wav'
    cmd = ['ffmpeg', '-i', str(source_file),
           '-ss', str(start_ms / 1000.0),
           '-t',  str((end_ms - start_ms) / 1000.0),
           '-acodec', 'pcm_s16le', '-ar', str(sr), '-ac', '1', '-y', str(out)]
    try:
        subprocess.run(cmd, check=True, capture_output=True, timeout=30)
        return out
    except Exception as e:
        print(f'    ffmpeg extract error (seg {idx}): {e}')
        return None


def _generate_silence(duration_ms: float, tmp_dir: str, idx: int,
                      sr: int = 16000) -> Optional[Path]:
    out = Path(tmp_dir) / f'sil_{idx:04d}.wav'
    cmd = ['ffmpeg', '-f', 'lavfi',
           '-i', f'anullsrc=r={sr}:cl=mono',
           '-t',  str(duration_ms / 1000.0),
           '-acodec', 'pcm_s16le', '-ar', str(sr), '-ac', '1', '-y', str(out)]
    try:
        subprocess.run(cmd, check=True, capture_output=True, timeout=10)
        return out
    except Exception as e:
        print(f'    ffmpeg silence error: {e}')
        return None


def _concat_wavs(wav_files: List[Path], output_path: Path,
                 sr: int = 16000) -> bool:
    if not wav_files:
        return False
    if len(wav_files) == 1:
        shutil.copy(wav_files[0], output_path)
        return True
    inputs = []
    for f in wav_files:
        inputs += ['-i', str(f)]
    filt = ''.join(f'[{i}:0]' for i in range(len(wav_files)))
    filt += f'concat=n={len(wav_files)}:v=0:a=1[out]'
    cmd = (['ffmpeg'] + inputs +
           ['-filter_complex', filt, '-map', '[out]',
            '-acodec', 'pcm_s16le', '-ar', str(sr), '-ac', '1',
            '-y', str(output_path)])
    try:
        subprocess.run(cmd, check=True, capture_output=True, timeout=60)
        return True
    except Exception as e:
        print(f'    ffmpeg concat error: {e}')
        return False


def _compute_relative_word_times(source_segs: List) -> Tuple[List[Dict], float]:
    """Recalcule les timestamps word-level relatifs au début du fichier produit."""
    cursor_ms    = 0.0
    word_times_r = []

    for k, seg in enumerate(source_segs):
        offset_ms = cursor_ms - seg.mfa_start_ms
        for wt in seg.word_times:
            word_times_r.append({
                'word'    : wt['word'],
                'start_ms': round(wt['start_ms'] + offset_ms, 1),
                'end_ms'  : round(wt['end_ms']   + offset_ms, 1),
            })
        cursor_ms += seg.mfa_duration_ms
        if k < len(source_segs) - 1:
            gap_ms     = max(0.0, source_segs[k + 1].mfa_start_ms - seg.mfa_end_ms)
            cursor_ms += gap_ms

    return word_times_r, cursor_ms


def _build_silence_padded_audio(source_segs: List, output_path: Path,
                                 sr: int = 16000) -> Tuple[bool, List[Dict], float]:
    """Construit l'audio silence-padded : speech du speaker + silence aux gaps."""
    word_times_r, total_dur = _compute_relative_word_times(source_segs)

    with tempfile.TemporaryDirectory() as tmp:
        parts: List[Path] = []
        for k, seg in enumerate(source_segs):
            seg_wav = _extract_to_temp(seg.audio_file, seg.mfa_start_ms,
                                       seg.mfa_end_ms, tmp, k * 2, sr)
            if seg_wav is None:
                return False, [], 0.0
            parts.append(seg_wav)

            if k < len(source_segs) - 1:
                gap_ms = max(0.0, source_segs[k + 1].mfa_start_ms - seg.mfa_end_ms)
                if gap_ms > 1:
                    sil = _generate_silence(gap_ms, tmp, k * 2 + 1, sr)
                    if sil:
                        parts.append(sil)

        ok = _concat_wavs(parts, output_path, sr)
        return ok, word_times_r, total_dur


# ─── Algorithme de merge greedy ──────────────────────────────────────────────

def merge_segments_by_speaker(
    audio_segment_metas: List[AudioSegmentMeta],
    output_dir: Path,
    target_duration_sec: Tuple[float, float] = (10.0, 30.0),
    sample_rate: int = 16000,
    verbose: bool = True
) -> List[MergedSegment]:
    """
    Merge contrôlé par speaker avec silence-padding.
    Ne traverse jamais une frontière de speaker ni de fichier source.
    Stratégie greedy : accumule jusqu'à min, flush quand max serait dépassé.
    """
    min_ms = target_duration_sec[0] * 1000
    max_ms = target_duration_sec[1] * 1000

    audio_out = Path(output_dir) / 'audio'
    audio_out.mkdir(parents=True, exist_ok=True)

    # ── Groupes consécutifs par (file_name, speaker) ─────────────────────────
    groups = []
    i = 0
    while i < len(audio_segment_metas):
        cur = audio_segment_metas[i]
        key = (cur.file_name, cur.speaker)
        grp, j = [cur], i + 1
        while j < len(audio_segment_metas):
            nxt = audio_segment_metas[j]
            if (nxt.file_name, nxt.speaker) == key:
                grp.append(nxt); j += 1
            else:
                break
        groups.append((cur.file_name, cur.speaker, cur.rel_path, grp))
        i = j

    merged_list: List[MergedSegment] = []
    merge_ctr   = 0
    skipped     = 0

    def _flush(buf: List, ctr: int) -> Tuple[Optional[MergedSegment], int]:
        nonlocal skipped
        if not buf:
            return None, ctr
        speech_ms = sum(s.mfa_duration_ms for s in buf)
        if speech_ms < min_ms:
            skipped += 1
            if verbose:
                ids = ' | '.join(s.segment_id for s in buf)
                print(f'    ⚠️  too short ({speech_ms/1000:.2f}s < {min_ms/1000:.0f}s): {ids}')
            return None, ctr
        gaps_ms   = sum(max(0.0, buf[k+1].mfa_start_ms - buf[k].mfa_end_ms)
                        for k in range(len(buf) - 1))
        total_dur = speech_ms + gaps_ms
        merge_id  = f'merged_{buf[0].file_name}_{buf[0].speaker}_{ctr:04d}'
        return MergedSegment(
            merge_id=merge_id,
            speaker=buf[0].speaker,
            file_name=buf[0].file_name,
            rel_path=buf[0].rel_path,
            original_segment_ids=[s.segment_id for s in buf],
            texts=[s.text for s in buf],
            merged_text=' '.join(s.text for s in buf),
            merged_duration_ms=total_dur,
            word_times_merged=[],
            source_segments=list(buf),
            audio_source_file=buf[0].audio_file,
        ), ctr + 1

    for file_name, speaker, rel_path, group in groups:
        buf, buf_ms = [], 0.0
        for seg in group:
            dur = seg.mfa_duration_ms
            if buf and (buf_ms + dur > max_ms) and (buf_ms >= min_ms):
                m, merge_ctr = _flush(buf, merge_ctr)
                if m: merged_list.append(m)
                buf, buf_ms = [], 0.0
            buf.append(seg)
            buf_ms += dur
            if buf_ms >= min_ms:
                m, merge_ctr = _flush(buf, merge_ctr)
                if m: merged_list.append(m)
                buf, buf_ms = [], 0.0
        if buf:
            m, merge_ctr = _flush(buf, merge_ctr)
            if m: merged_list.append(m)

    # ── Générer les fichiers audio ────────────────────────────────────────────
    print(f'\n   Génération de {len(merged_list)} fichiers audio silence-padded...')
    success = 0
    for merged in tqdm(merged_list, desc='Merge audio'):
        spk_dir = audio_out / merged.speaker
        spk_dir.mkdir(exist_ok=True)
        audio_path = spk_dir / f'{merged.merge_id}.wav'

        ok, wt_rel, total_dur = _build_silence_padded_audio(
            merged.source_segments, audio_path, sample_rate
        )
        if ok:
            merged.merged_audio_path = audio_path
            merged.word_times_merged = wt_rel
            merged.merged_duration_ms = total_dur
            success += 1

    # ── Rapport ───────────────────────────────────────────────────────────────
    print('\n' + '=' * 70)
    print('ZONE 5 : RAPPORT MERGE')
    print('=' * 70)
    print(f'   Segments fusionnés créés : {len(merged_list)}')
    print(f'   Audio générés            : {success}/{len(merged_list)}')
    print(f'   Groupes ignorés (<{target_duration_sec[0]:.0f}s) : {skipped}')

    if merged_list:
        total_durs  = [m.duration_sec for m in merged_list if m.merged_audio_path]
        speech_durs = [sum(s.mfa_duration_ms for s in m.source_segments) / 1000.0
                       for m in merged_list if m.merged_audio_path]
        if total_durs:
            print(f'\n   Durée totale (speech+silence) : '
                  f'moy {np.mean(total_durs):.1f}s | min {np.min(total_durs):.1f}s | max {np.max(total_durs):.1f}s')
            print(f'   Durée speech seule           : '
                  f'moy {np.mean(speech_durs):.1f}s | min {np.min(speech_durs):.1f}s | max {np.max(speech_durs):.1f}s')
            in_range = sum(1 for d in speech_durs
                          if target_duration_sec[0] <= d <= target_duration_sec[1])
            print(f'   Dans [{target_duration_sec[0]:.0f}s, {target_duration_sec[1]:.0f}s] (speech) : '
                  f'{in_range}/{len(merged_list)} ({100*in_range/len(merged_list):.1f}%)')
        print(f'   Par speaker : {dict(Counter(m.speaker for m in merged_list))}')

    return merged_list


print('\n' + '=' * 70)
print('ZONE 5 : MERGE CONTRÔLÉ PAR SPEAKER (10–30 s)')
print('=' * 70)

merged_segments = merge_segments_by_speaker(
    audio_segment_metas=audio_segment_metas,
    output_dir=CONFIG['merged_dir'],
    target_duration_sec=(CONFIG['merge_min_sec'], CONFIG['merge_max_sec']),
    sample_rate=CONFIG['sample_rate'],
    verbose=True
)

---
## Zone 6 : Quality Analysis

Score 0–1 basé sur les features acoustiques et linguistiques. Dashboard de visualisation.

In [ ]:
try:
    import librosa
except ImportError:
    import subprocess
    subprocess.run(['pip', 'install', '-q', 'librosa'], check=True)
    import librosa

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt


def compute_quality_score(row: Dict) -> Tuple[float, List[str]]:
    """
    Score de qualité rule-based [0, 1].
    Pénalités cumulatives pour chaque règle violée.
    """
    score  = 1.0
    issues = []

    # R1 : speech rate (0.5 – 6.0 wps)
    rs = row.get('speech_rate_wps', 0)
    if not (0.5 <= rs <= 6.0):
        score -= 0.30; issues.append('speech rate outside range (0.5–6.0 wps)')

    # R2 : durée (5 – 60 s)
    dur_s = row.get('duration_ms', 0) / 1000.0
    if not (5.0 <= dur_s <= 60.0):
        score -= 0.20; issues.append('duration outside range (5–60 s)')

    # R3 : longueur texte (>= 10 chars)
    if row.get('text_length', 0) < 10:
        score -= 0.10; issues.append('text too short (< 10 chars)')

    # R4 : speech activity (>= 0.30)
    if row.get('speech_activity_ratio', 1) < 0.30:
        score -= 0.25; issues.append('low speech activity (< 30%)')

    # R5 : énergie moyenne (>= 0.01)
    if row.get('energy_mean', 1) < 0.01:
        score -= 0.10; issues.append('audio too quiet (energy < 0.01)')

    # R6 : dynamic range (>= 5 dB)
    if row.get('dynamic_range_db', 10) < 5.0:
        score -= 0.15; issues.append('poor dynamic range (< 5 dB)')

    return max(0.0, min(1.0, score)), issues


def extract_features(merged_segs: List[MergedSegment],
                     sr: int = 16000) -> pd.DataFrame:
    """Extrait les features acoustiques + linguistiques pour chaque segment."""
    rows = []
    for seg in tqdm(merged_segs, desc='Features'):
        if not seg.merged_audio_path or not Path(seg.merged_audio_path).exists():
            continue
        row = {
            'segment_id'      : seg.merge_id,
            'speaker'         : seg.speaker,
            'duration_ms'     : seg.merged_duration_ms,
            'text'            : seg.merged_text,
            'n_words'         : seg.num_words,
            'text_length'     : len(seg.merged_text),
        }
        if seg.merged_duration_ms > 0:
            row['speech_rate_wps'] = seg.num_words / (seg.merged_duration_ms / 1000.0)
        else:
            row['speech_rate_wps'] = 0.0

        try:
            y, _ = librosa.load(str(seg.merged_audio_path), sr=sr, mono=True)
            if len(y) > 0:
                rms   = librosa.feature.rms(y=y, frame_length=512, hop_length=512)[0]
                thr   = np.percentile(rms, 30)
                row['energy_mean']            = float(np.mean(rms))
                row['speech_activity_ratio']  = float(np.mean(rms > thr))
                S     = np.abs(librosa.stft(y))
                if S.max() > 0:
                    row['dynamic_range_db'] = float(20 * np.log10(
                        (S.max() + 1e-10) / (S[S > 0].min() + 1e-10)
                    ))
                else:
                    row['dynamic_range_db'] = 0.0
        except Exception:
            row.update({'energy_mean': 0.0, 'speech_activity_ratio': 0.0,
                        'dynamic_range_db': 0.0})

        score, issues = compute_quality_score(row)
        row['quality_score']  = score
        row['quality_issues'] = issues
        rows.append(row)

    return pd.DataFrame(rows)


def plot_quality_dashboard(df: pd.DataFrame, save_path: Optional[Path] = None):
    """Dashboard 2×3 : distribution score, speech rate vs quality, durée,
       mots/segment, speech activity, qualité par speaker."""
    if df.empty:
        print('⚠️  DataFrame vide — pas de visualisation'); return

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle('Quality Analysis Dashboard', fontsize=16, fontweight='bold')

    axes[0,0].hist(df['quality_score'], bins=20, color='#2E86AB', alpha=0.7, edgecolor='black')
    axes[0,0].axvline(df['quality_score'].mean(), color='red', linestyle='--', label=f'Mean {df["quality_score"].mean():.3f}')
    axes[0,0].set_title('Quality Score Distribution'); axes[0,0].legend(); axes[0,0].grid(True, alpha=0.3)

    if 'speech_rate_wps' in df.columns:
        sc = axes[0,1].scatter(df['speech_rate_wps'], df['quality_score'],
                               c=df['quality_score'], cmap='RdYlGn', alpha=0.6)
        axes[0,1].set_xlabel('Speech Rate (wps)'); axes[0,1].set_ylabel('Quality Score')
        axes[0,1].set_title('Speech Rate vs Quality'); axes[0,1].grid(True, alpha=0.3)

    axes[0,2].hist(df['duration_ms'] / 1000, bins=40, color='#A23B72', alpha=0.7, edgecolor='black')
    axes[0,2].set_xlabel('Duration (s)'); axes[0,2].set_title('Segment Duration'); axes[0,2].grid(True, alpha=0.3, axis='y')

    axes[1,0].hist(df['n_words'], bins=40, color='#F18F01', alpha=0.7, edgecolor='black')
    axes[1,0].set_xlabel('Nb mots'); axes[1,0].set_title('Mots / segment'); axes[1,0].grid(True, alpha=0.3, axis='y')

    if 'speech_activity_ratio' in df.columns:
        axes[1,1].hist(df['speech_activity_ratio'].dropna(), bins=40, color='#06A77D', alpha=0.7, edgecolor='black')
        axes[1,1].axvline(0.30, color='red', linestyle='--', label='Seuil 30%')
        axes[1,1].set_xlabel('Speech Activity Ratio'); axes[1,1].set_title('VAD Activity'); axes[1,1].legend(); axes[1,1].grid(True, alpha=0.3, axis='y')

    if 'speaker' in df.columns:
        spk_q = df.groupby('speaker')['quality_score'].mean().sort_values(ascending=False).head(10)
        axes[1,2].barh(range(len(spk_q)), spk_q.values, color='#2E86AB', alpha=0.7, edgecolor='black')
        axes[1,2].set_yticks(range(len(spk_q))); axes[1,2].set_yticklabels(spk_q.index)
        axes[1,2].set_xlabel('Avg Quality Score'); axes[1,2].set_title('Qualité par Speaker'); axes[1,2].grid(True, alpha=0.3, axis='x')

    plt.tight_layout()
    if save_path:
        plt.savefig(str(save_path), dpi=150, bbox_inches='tight')
        print(f'   📊 Dashboard sauvegardé : {save_path}')
    plt.show()


print('=' * 70)
print('ZONE 6 : QUALITY ANALYSIS')
print('=' * 70)

features_df = extract_features(merged_segments, sr=CONFIG['sample_rate'])

if not features_df.empty:
    print(f'\n   Segments analysés   : {len(features_df)}')
    print(f'   Score moyen         : {features_df["quality_score"].mean():.3f}')
    print(f'   Score médian        : {features_df["quality_score"].median():.3f}')

    threshold = CONFIG['quality_threshold']
    kept    = features_df[features_df['quality_score'] >= threshold]
    removed = features_df[features_df['quality_score'] <  threshold]
    print(f'\n   ✅ Gardés  (>= {threshold:.2f}) : {len(kept)} ({100*len(kept)/len(features_df):.1f}%)')
    print(f'   ❌ Retirés (<  {threshold:.2f}) : {len(removed)} ({100*len(removed)/len(features_df):.1f}%)')

    if len(removed) > 0:
        issue_counts = {}
        for issues_list in removed['quality_issues']:
            for issue in issues_list:
                issue_counts[issue] = issue_counts.get(issue, 0) + 1
        print('\n   Raisons de rejet :')
        for issue, cnt in sorted(issue_counts.items(), key=lambda x: -x[1]):
            print(f'      • {issue:<55}: {cnt}')

    dash_path = CONFIG['output_jsonl'].parent / 'quality_dashboard.png'
    plot_quality_dashboard(features_df, save_path=dash_path)
    print(f'\n✅ ZONE 6 COMPLETE — {len(kept)} segments retenus')
else:
    print('⚠️  Aucun segment à analyser')
    kept = pd.DataFrame()

---
## Zone 7 : Export JSONL

Exporte les segments fusionnés (filtrés par qualité) au format attendu par `04_whisper_preprocessing_augmentation_pipeline`.

In [ ]:
def export_merged_to_jsonl(
    merged_segments: List[MergedSegment],
    quality_df: pd.DataFrame,
    out_path: Path,
    quality_threshold: float = 0.70
) -> None:
    """
    Exporte les MergedSegments au format JSONL attendu par le pipeline 04.
    Filtre par quality_score si quality_df est fourni.

    Format par ligne :
        {
            'id'          : 'merged_01-1a_KAT_0003',
            'audio_path'  : '/path/merged.wav',
            'text'        : 'texte fusionné',
            'speaker'     : 'KAT',
            'source_file' : '01-1a',
            'source_audio': '/path/source.mp3',
            'duration_sec': 18.34,
            'num_words'   : 22,
            'word_times'  : [...],
        }
    """
    print('=' * 70)
    print('ZONE 7 : EXPORT JSONL')
    print('=' * 70)

    # Construire l'ensemble des IDs de qualité suffisante
    if not quality_df.empty:
        good_ids = set(quality_df[quality_df['quality_score'] >= quality_threshold]['segment_id'])
        print(f'\n   Filtre qualité >= {quality_threshold:.2f} : {len(good_ids)} / {len(quality_df)} segments')
    else:
        # Pas de quality_df → exporter tout ce qui a un audio
        good_ids = {m.merge_id for m in merged_segments if m.merged_audio_path}
        print(f'\n   Pas de filtre qualité — export de tous les segments ({len(good_ids)})')

    exported = 0
    skipped  = 0

    with open(out_path, 'w', encoding='utf-8') as f:
        for m in merged_segments:
            if m.merge_id not in good_ids:
                skipped += 1
                continue
            if not m.merged_audio_path or not Path(m.merged_audio_path).exists():
                skipped += 1
                continue
            f.write(json.dumps(m.to_jsonl_dict(), ensure_ascii=False) + '\n')
            exported += 1

    print(f'   ✅ Exportés : {exported}')
    print(f'   ⏭️  Ignorés  : {skipped} (qualité insuffisante ou audio manquant)')
    print(f'\n   📄 Fichier JSONL : {out_path}')
    print(f'   ➡️  Passer ce fichier à : 04_whisper_preprocessing_augmentation_pipeline.ipynb')
    print(f'      CONFIG["input_jsonl"] = Path("{out_path}")')


export_merged_to_jsonl(
    merged_segments=merged_segments,
    quality_df=features_df if 'features_df' in dir() and not features_df.empty else pd.DataFrame(),
    out_path=CONFIG['output_jsonl'],
    quality_threshold=CONFIG['quality_threshold']
)

# ── Rapport CSV de qualité (pour audit) ──────────────────────────────────────
if 'features_df' in dir() and not features_df.empty:
    report_path = CONFIG['output_jsonl'].parent / 'merge_quality_report.csv'
    features_df.drop(columns=['quality_issues'], errors='ignore').to_csv(report_path, index=False)
    print(f'   📊 Rapport CSV : {report_path}')